# Module 4 — Document Chat Bot (Long Context Version)

This notebook demonstrates a document Q&A chatbot using the LLM's long context window (~1M tokens).

### How it works

- User uploads documents (.md, .txt, .docx)
- All documents are concatenated into one long text
- The LLM reads the full text via its large context window
- User asks questions, LLM answers from the document content

### Requirements

- A model with large context window (DeepSeek ~1M, Gemini ~1M, Claude ~200K)
- API Key configured in `.env` file

---


## 1. Environment Setup

Select Kernel: "Python (deep_agent_0530)"


In [7]:
from __future__ import annotations

import io
import os
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import ipywidgets as widgets
from dotenv import load_dotenv
from IPython.display import display, clear_output
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent

MODULE_DIR = Path.cwd()
PROJECT_ROOT = MODULE_DIR.parents[1]
ARTIFACTS_DIR = MODULE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")

print("MODULE_DIR =", MODULE_DIR)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("ARTIFACTS_DIR =", ARTIFACTS_DIR)


MODULE_DIR = /Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/notebooks/module-4
PROJECT_ROOT = /Users/weiping/dev/Learn/langchain-ai/deep_agent_0530
ARTIFACTS_DIR = /Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/notebooks/module-4/artifacts


### Initialize LLM

Configure via environment variables:

- `DOCBOT_MODEL_PROVIDER` (default: deepseek)
- `DOCBOT_MODEL_NAME` (default: deepseek-chat)
- `DOCBOT_MODEL_TEMPERATURE` (default: 0.3)


In [8]:
def build_model():
    provider = os.environ.get("DOCBOT_MODEL_PROVIDER", "deepseek")
    model_name = os.environ.get("DOCBOT_MODEL_NAME", "deepseek-chat")
    temperature = float(os.environ.get("DOCBOT_MODEL_TEMPERATURE", "0.3"))
    llm = init_chat_model(
        model=model_name,
        model_provider=provider,
        temperature=temperature,
    )
    return llm, provider, model_name

llm, llm_provider, llm_model_name = build_model()
print(f"Model: {llm_model_name}  Provider: {llm_provider}")


Model: deepseek-chat  Provider: deepseek


## 2. Upload Documents

Supported formats: `.md`, `.txt`, `.docx`

Files are saved to `artifacts/` directory.


In [9]:
upload_btn = widgets.FileUpload(
    accept=".md,.txt,.docx",
    multiple=True,
    description="Upload Documents",
    button_style="primary",
    layout=widgets.Layout(width="auto"),
)

upload_output = widgets.Output()

def docx_to_markdown(docx_bytes: bytes) -> str:
    lines = []
    with zipfile.ZipFile(io.BytesIO(docx_bytes)) as z:
        xml_content = z.read("word/document.xml")
    root = ET.fromstring(xml_content)
    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
    for para in root.iter("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}p"):
        texts = []
        for t in para.iter("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}t"):
            if t.text:
                texts.append(t.text)
        text = "".join(texts).strip()
        if not text:
            lines.append("")
            continue
        ppr = para.find(".//w:pStyle", ns)
        if ppr is not None:
            style_val = ppr.get("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val", "")
            if style_val.startswith("Heading"):
                try:
                    level = int(style_val.replace("Heading", ""))
                except ValueError:
                    level = 1
                lines.append(f"{'#' * level} {text}")
                continue
        lines.append(text)
    return "\n".join(lines)

def on_upload(change):
    with upload_output:
        clear_output(wait=True)
        uploaded = change["new"]
        if not uploaded:
            return
        for filename, file_info in uploaded.items():
            content = file_info["content"]
            if isinstance(content, bytes):
                if filename.endswith(".docx"):
                    content = docx_to_markdown(content)
                else:
                    content = content.decode("utf-8")
            dest = ARTIFACTS_DIR / filename
            dest.write_text(content, encoding="utf-8")
            print(f"Saved: {filename} -> {dest.name}  ({len(content):,} chars)")

upload_btn.observe(on_upload, names="value")

display(upload_btn, upload_output)


FileUpload(value=(), accept='.md,.txt,.docx', button_style='primary', description='Upload Documents', layout=L…

Output()

## 3. Process Documents

Read all uploaded files and concatenate into one text for the LLM context.


In [ ]:
def load_documents() -> str:
    parts = []
    supported = (".md", ".txt", ".docx")
    files = sorted(ARTIFACTS_DIR.iterdir())
    if not files:
        print("artifacts/ is empty, please upload documents first.")
        return ""

    for f in files:
        if f.suffix not in supported or not f.is_file():
            continue
        print(f"  Reading: {f.name}...", end=" ")
        if f.suffix == ".docx":
            text = docx_to_markdown(f.read_bytes())
        else:
            text = f.read_text(encoding="utf-8")
        parts.append(text)
        print(f"{len(text):,} chars")

    doc_text = "\n\n---\n\n".join(parts)
    print(f"\nTotal document length: {len(doc_text):,} chars")
    estimated_tokens = len(doc_text) // 4
    print(f"Estimated tokens: ~{estimated_tokens:,}")
    return doc_text

document_text = load_documents()


### Document Structure Preview

Show first 500 chars and section headings.


In [ ]:
if document_text:
    print("=" * 60)
    print("Document preview (first 500 chars):")
    print("=" * 60)
    print(document_text[:500])
    print()
    import re
    headings = re.findall(r"^#{1,4}\s+.+", document_text, re.MULTILINE)
    if headings:
        print("Document sections:")
        for h in headings[:30]:
            print(f"  {h}")


## 4. Build Q&A Agent

A LangGraph agent with `read_uploaded_documents` tool to access the full document text.


In [ ]:
if not document_text:
    print("Document is empty, please upload first.")

@tool
def read_uploaded_documents() -> str:
    """Read all uploaded documents. Call this first before answering questions."""
    return document_text

tools = [read_uploaded_documents]

SYSTEM_PROMPT = "You are a document Q&A assistant.\n\n" \
    "1. When the user asks a question, first call read_uploaded_documents to get the full document content.\n" \
    "2. Answer based on the document content.\n" \
    "3. If the answer is not in the documents, say so.\n" \
    "4. Cite specific parts of the document when possible.\n" \
    "5. For summary/analysis questions, read thoroughly and give structured answers."

agent = create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)
print("Agent created successfully")


## 5. Ask a Question

Enter your question below. The agent reads the document and answers.


In [ ]:
question = input("Enter your question: ")
if question.strip():
    print(f"\nYou: {question}\n")
    print("Assistant: ", end="", flush=True)
    chat(question)
else:
    print("Please enter a valid question.")


### Multi-turn Chat

Continuous conversation with history. Type `exit` to quit.


In [ ]:
print("Multi-turn chat mode (type exit to quit)\n")
while True:
    q = input("\nYou: ")
    if q.strip().lower() in ("exit", "quit", "q"):
        print("\nGoodbye!")
        break
    if not q.strip():
        continue
    print("Assistant: ", end="", flush=True)
    chat(q)


## 6. Token Usage & Cost Report


In [ ]:
print("=" * 80)
print("  Token Usage & Cost Report")
print("=" * 80)

model_provider = os.environ.get("DOCBOT_MODEL_PROVIDER", "deepseek")
model_name = os.environ.get("DOCBOT_MODEL_NAME", "deepseek-chat")

PRICING = {
    "deepseek": {"input": 0.5, "output": 2.0},
    "openai":    {"input": 2.5, "output": 10.0},
    "anthropic": {"input": 3.0, "output": 15.0},
    "minimax":   {"input": 0.5, "output": 2.0},
}
pricing = PRICING.get(model_provider, PRICING["deepseek"])

print(f"Model Provider : {model_provider}")
print(f"Model Name     : {model_name}")
print(f'Input Price    : ¥{pricing["input"]}/1M tokens')
print(f'Output Price   : ¥{pricing["output"]}/1M tokens')
print()

input_tokens = len(document_text) // 4 if document_text else 0
output_tokens = 0

input_cost = (input_tokens / 1_000_000) * pricing["input"]
output_cost = (output_tokens / 1_000_000) * pricing["output"]
total_cost = input_cost + output_cost

print("=" * 60)
print(f"  {'Item':<30} {'Count':>12}")
print("=" * 60)
print(f"  {'Document Chars':<30} {len(document_text):>12,}")
print(f"  {'Estimated Input Tokens':<30} {input_tokens:>12,}")
print("-" * 60)
print(f"  {'Input Cost (¥)':<30} {input_cost:>12.6f}")
print(f"  {'Output Cost (¥)':<30} {output_cost:>12.6f}")
print(f"  {'Total Cost (¥)':<30} {total_cost:>12.6f}")
print("=" * 60)
print()
print("Note: Each question sends the full document + conversation history.")
print("Multiple questions will accumulate costs.")
